# Study 876 — Industry-Relative MAX 🎰

**Does adjusting a stock's MAX for its sector sharpen — or kill — the lottery effect?**

The lottery / MAX effect (study 365, Bali-Cakici-Whitelaw 2011) sorts a name on its own
**maximum daily return** last month and finds the lottery-like high-MAX names *under-earn*.
But part of a name's MAX is just **sector-wide volatility** — a whole sector can be jumpy for
macro reasons — which is noise for a *lottery-demand* story. Here we subtract the **median MAX
of the name's sector peers** to get the **industry-relative** MAX, a cleaner proxy for
*idiosyncratic* lottery demand, and ask whether the negative MAX→return relation sharpens.

We take the self-contained monthly version on a liquid US cross-section (2010-01-04 →
2026-06-30, 50 names across 8 GICS sectors).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper bound.*


## 1. The idea in one picture

A stock's MAX (its biggest one-day pop) is two things stacked: **sector weather** (the whole sector was jumpy) plus a **name-specific lottery pop**. The lottery story is about the *idiosyncratic* pop, so we strip the sector part out: `industry-relative MAX = own MAX − median MAX of sector peers`. Then sort, buy the boring low-MAX tail, sell the lottery high-MAX tail — and check whether the cleaner signal pays better.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_sectors': 8, 'n_months': 198, 'n_spread': 197, 'raw_bps': -104.8, 'raw_t': -2.42, 'raw_t1s': -2.53, 'raw_win': 43, 'raw_sharpe': -0.62, 'raw_p': 0.994, 'adj_bps': -89.7, 'adj_t': -2.51, 'adj_t1s': -2.8, 'adj_win': 46, 'adj_sharpe': -0.69, 'adj_p': 0.998, 'q1': 17.5, 'q2': 13.1, 'q3': 15.45, 'q4': 16.95, 'q5': 28.26, 'placebo_obs': -89.7, 'placebo_mean': -0.19, 'placebo_sd': 32.56, 'placebo_sigma_left': -2.75, 'placebo_left_p': 0.0023, 'placebo_draws': 20000, 'era_early_bps': -47.1, 'era_early_t': -1.26, 'era_early_n': 96, 'era_late_bps': -130.2, 'era_late_t': -2.24, 'era_late_n': 101, 'timer_1_gross': -89.7, 'timer_1_cost': 6.2, 'timer_1_net': -95.9, 'timer_1_t': -3.0, 'timer_1_ann': -11.5, 'timer_5_gross': -89.7, 'timer_5_cost': 14.2, 'timer_5_net': -103.9, 'timer_5_t': -3.25, 'timer_5_ann': -12.5, 'null_mean_t': -0.25, 'null_sd_t': 0.88, 'null_fire': 1, 'planted_raw_t': 11.18, 'planted_adj_t': 20.6, 'fingerprint': '5384b8f7f128'}
print('RAW MAX          : spread %+.1f bps/mo  (NW t = %+.2f)' % (R['raw_bps'], R['raw_t']))
print('INDUSTRY-RELATIVE: spread %+.1f bps/mo  (NW t = %+.2f)' % (R['adj_bps'], R['adj_t']))
print()
print('Both spreads are NEGATIVE -> the claim INVERTS: on mega-caps the lottery')
print('(high-MAX) names OUT-earned the boring low-MAX ones. The industry')
print('adjustment does not fix the sign; it slightly sharpens the wrong-sign t.')

RAW MAX          : spread -104.8 bps/mo  (NW t = -2.42)
INDUSTRY-RELATIVE: spread -89.7 bps/mo  (NW t = -2.51)

Both spreads are NEGATIVE -> the claim INVERTS: on mega-caps the lottery
(high-MAX) names OUT-earned the boring low-MAX ones. The industry
adjustment does not fix the sign; it slightly sharpens the wrong-sign t.


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world where each name's MAX = an **un-priced sector-wide level** + a **priced idiosyncratic pop** (`edge>0`). The industry adjustment removes the sector level, so it should recover the planted relation *more sharply* than the raw MAX — and both must stay silent on the null (`edge=0`). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from max_industry import data, strategy as st
planted = data.synthetic_panel(edge=0.012, seed=876, n_months=240)
null = data.synthetic_panel(edge=0.0, seed=876, n_months=240)
print('planted world  raw-MAX  NW t = %+.2f' % st.synthetic_detect(planted, adjusted=False)['t_nw'])
print('planted world  ind-rel  NW t = %+.2f  (adjustment SHARPENS)' % st.synthetic_detect(planted, adjusted=True)['t_nw'])
print('null world     ind-rel  NW t = %+.2f  (should be ~0)' % st.synthetic_detect(null, adjusted=True)['t_nw'])

planted world  raw-MAX  NW t = +11.18


planted world  ind-rel  NW t = +20.60  (adjustment SHARPENS)


null world     ind-rel  NW t = +1.53  (should be ~0)


## 3. The honest verdict — a cleaner knife on a claim that isn't there

On this liquid mega-cap tape the industry-relative long-low / short-high MAX spread is **-89.7 bps/mo** with NW *t* = **-2.51** — significant, but with the **opposite sign** to the MAX effect: here the lottery high-MAX names actually *out-earned* the boring low-MAX ones (the sign-flip placebo puts the observed value ~2.8σ into the *left* tail). The industry adjustment barely moves the raw *t* (-2.42 → -2.51): it sharpens the knife but there is no (right-signed) apple to cut. The seeded synthetic control recovers a *planted* relation and confirms the adjustment sharpens it, so this is a genuine sign-reversal on the mega-cap survivor universe, not a bug — the MAX premium is a small-and-illiquid-stock phenomenon. **Signal: None** (the claimed edge is absent — and inverts), **Tradability: Mirage** (the specified book loses money gross and net).